# Deterministic Calendar Scheduling with MCP

LLMs score below 50% on temporal reasoning tasks ([OOLONG benchmark](https://arxiv.org/abs/2511.02817)). Earlier models scored as low as 29% on scheduling and 13% on duration calculations ([Test of Time, ICLR 2025](https://arxiv.org/abs/2406.09170)). Ask "Schedule for next Tuesday at 2pm" and the model picks the wrong Tuesday. Ask "Am I free at 3pm?" and it checks the wrong timezone. Then it double-books your calendar.

This notebook shows how to build a scheduling agent that **never hallucinates dates** and **never double-books** — by connecting the OpenAI Agents SDK to a calendar MCP server ([Temporal Cortex](https://github.com/temporal-cortex/mcp)) via `HostedMCPTool`. The MCP server provides deterministic datetime resolution, cross-provider availability merging, and atomic booking with Two-Phase Commit.

## Setup

You'll need:
- An **OpenAI API key** with Responses API access ([platform.openai.com](https://platform.openai.com/api-keys))
- A **Temporal Cortex API key** ([app.temporal-cortex.com](https://app.temporal-cortex.com)) with at least one calendar connected

In [ ]:
%pip install openai-agents python-dotenv -q

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

# Set your API keys (or use a .env file)
# os.environ["OPENAI_API_KEY"] = "sk-..."
# os.environ["TEMPORAL_CORTEX_API_KEY"] = "sk_live_..."

## Step 1: Connect to the Calendar MCP Server

`HostedMCPTool` delegates the MCP connection to OpenAI's Responses API. You provide the server URL and auth headers — OpenAI connects, discovers tools, and handles tool invocation server-side. No local MCP process needed.

In [ ]:
from agents import Agent, HostedMCPTool, Runner

temporal_cortex = HostedMCPTool(
    tool_config={
        "type": "mcp",
        "server_label": "temporal-cortex",
        "server_url": "https://mcp.temporal-cortex.com/mcp",
        "headers": {
            "Authorization": f"Bearer {os.getenv('TEMPORAL_CORTEX_API_KEY', '')}",
        },
        "require_approval": "never",
    }
)

print("HostedMCPTool configured — OpenAI will connect to the MCP server on first use.")

## Step 2: Build a Scheduling Agent

The agent instructions encode a deterministic workflow: orient in time → resolve datetime → discover calendars → find availability → book. This prevents the model from guessing dates or skipping availability checks.

We start with a **read-only query** (availability check) so the agent doesn't modify your calendar. The next section adds approval-gated booking.

In [ ]:
agent = Agent(
    name="Calendar Scheduler",
    instructions=(
        "You schedule meetings using Temporal Cortex calendar tools.\n\n"
        "Follow this workflow:\n"
        "1. Call get_temporal_context to learn the current time and timezone\n"
        "2. Call resolve_datetime to convert human expressions to RFC 3339 timestamps\n"
        "3. Call list_calendars to discover connected calendars\n"
        "4. Call find_free_slots to check availability on the target date\n"
        "5. Call book_slot to book the meeting (Two-Phase Commit prevents double-bookings)\n\n"
        "When calling data tools (list_calendars, list_events, find_free_slots, "
        "expand_rrule, get_availability), pass format='json' for structured output.\n"
        "Always use provider-prefixed calendar IDs (e.g., google/primary).\n"
        "Never guess dates or times — always use the tools."
    ),
    tools=[temporal_cortex],
)

print(f"Agent '{agent.name}' ready with {len(agent.tools)} tool source(s).")

In [ ]:
result = await Runner.run(
    agent,
    "Am I free next Tuesday at 2pm? Check my calendar availability.",
)

print(result.final_output)

## Step 3: Add Approval Gates for Booking

Calendar booking is a write operation — you don't want the agent to book without confirmation. Use `require_approval` to gate specific tools while letting read-only tools run freely.

This maps to the MCP tool annotations: `book_slot` and `request_booking` have `readOnlyHint: false`, while all other tools are read-only. With approval gating, the agent can check availability autonomously but pauses before creating any calendar events.

In [ ]:
# Fine-grained approval: only booking tools require human sign-off.
# All read-only tools run without interruption.
gated_temporal_cortex = HostedMCPTool(
    tool_config={
        "type": "mcp",
        "server_label": "temporal-cortex",
        "server_url": "https://mcp.temporal-cortex.com/mcp",
        "headers": {
            "Authorization": f"Bearer {os.getenv('TEMPORAL_CORTEX_API_KEY', '')}",
        },
        "require_approval": {
            "book_slot": "always",
            "request_booking": "always",
        },
    }
)

gated_agent = Agent(
    name="Calendar Scheduler (Gated)",
    instructions=(
        "You schedule meetings using Temporal Cortex calendar tools.\n\n"
        "Follow this workflow:\n"
        "1. Call get_temporal_context to learn the current time\n"
        "2. Call resolve_datetime to convert the requested time\n"
        "3. Call list_calendars to discover calendars\n"
        "4. Call find_free_slots to check availability\n"
        "5. Present the available slots and your recommendation\n"
        "6. Call book_slot to book (this will require approval)\n\n"
        "Pass format='json' to data tools. Use provider-prefixed calendar IDs."
    ),
    tools=[gated_temporal_cortex],
)

print("Gated agent configured — book_slot and request_booking require approval.")
print("In production, wire the approval callback to a UI dialog or Slack message.")

## What Happened Under the Hood

When the agent ran, it followed the deterministic workflow encoded in its instructions:

1. **`get_temporal_context`** — The agent learned the current time, timezone, UTC offset, and DST status. This prevents timezone mistakes (e.g., scheduling in UTC when the user is in EST).

2. **`resolve_datetime`** — The expression "next Tuesday at 2pm" was resolved to a precise RFC 3339 timestamp. The MCP server uses deterministic date math — no LLM guessing which Tuesday is "next."

3. **`list_calendars` → `find_free_slots`** — The agent discovered connected calendar providers and checked actual availability, rather than assuming the time slot was open.

For the gated agent, **`book_slot`** uses Two-Phase Commit: acquire a time-range lock, verify no conflicts, write the event, release the lock. If two agents try to book the same slot simultaneously, exactly one succeeds.

The key insight: the agent **never guessed** any temporal value. Every date, time, and timezone came from deterministic tools. This is what brings scheduling accuracy from ~50% (LLM inference) to 100% (tool-assisted).

## Next Steps

- **Multi-agent workflows**: Use the Agent-as-Tool pattern to split temporal analysis, calendar queries, and booking across specialized agents. See [multi_agent.py](https://github.com/temporal-cortex/mcp/blob/main/examples/openai-agents/multi_agent.py).
- **Tool reference**: Full input/output schemas for all 15 tools at [temporal-cortex.com/docs](https://temporal-cortex.com/docs/tool-reference).
- **Agent Skills**: Procedural knowledge for calendar workflows at [github.com/temporal-cortex/skills](https://github.com/temporal-cortex/skills).
- **MCP guide**: General MCP integration patterns in the [Responses API MCP Tool Guide](https://developers.openai.com/cookbook/examples/mcp/mcp_tool_guide/).